# 05. Xe II: свой расчёт рассеяния в модели Ванга — проверка, метастабили, каскады

Три вещи, которых нет в ноутбуке 02 (там σ берутся готовыми у Ванга):

1. **Проверка σ Ванга.** Та же мишень (`models.xe2_wang`, 68 состояний, нумерация уровней —
   таблица Ванга), свой расчёт DBSR. Сравниваются σ и ⟨σv⟩ для **всех** пар, которые есть у
   Ванга (gs→6s/5d/6p, 6s→6p, 5d→6p). Это воспроизведение расчёта, а не другая физическая
   модель: совпадение проверяет, что Ванг и мы посчитали одно и то же (сетка, число
   парциальных волн, пороги). Зависимость от модели мишени — ноутбук 06.
2. **Переходы между метастабилями** (и с ²P₁/₂): у Ванга их нет. Метастабиль здесь — уровень
   ниже 6p, у которого время жизни по E1 (из этого же расчёта) длиннее `TAU_META` или
   E1-распада нет вовсе. M1 и E2 не считаются (dbsr_dmat3 падает на M1), поэтому настоящие
   времена жизни короче — проверьте по литературе, если важно.
3. **Каскады в верхние уровни выбранных линий.** Для каждого уровня h выше u — вероятность
   P(h→u), что h, распадаясь, пройдёт через u (все цепочки, ветвления DBSR E1). Константа
   каскадного заселения из уровня i: k_casc(i→u) = Σ_h k(i→h)·P(h→u). Отношение
   k_casc/k(i→u) — атомные данные (как σ_app у Fursa); как каскады войдут в заселённость, решает CRM.
   В модели Ванга выше 6p есть только 5d и 7s (до 18.5 эВ); 6d, 7p, 4f — ноутбук 06.

Все σ здесь — DBSR (этот расчёт); ветвления — DBSR E1 (этот расчёт).

**Время**: J = 0 (2 волны) на 4 ядрах — см. `docs/crm/lines_final.md`; полный расчёт
(J ≤ 25) — сервер. Сходимость по J проверьте (`JMAX` 15/20/25): у оптически разрешённых
переходов (gs→6s, 5d) большие J вносят заметный вклад.

**Выход** (папка готового расчёта `runs/xe_plus_v2`, если он есть, иначе `runs/crm/xe2_wang_full`): `sigma_xe2_wang.xlsx` (σ из начальных уровней Ванга,
формат Ванга), `rates_xe2_wang.xlsx` (k всех возбуждений), `check_vs_wang.csv`,
`xe2_levels_tau.csv`, `xe2_sigma_metastable.xlsx`, `xe2_k_metastable.xlsx`,
`xe2_cascade_P.csv`, `xe2_k_cascade.xlsx` (+ .csv).

In [ ]:
# ---------- параметры (поменяйте пути под себя)
from pathlib import Path
import sys
ROOT    = Path("/data/files_Otdel120/pydbsr_calc")    # папка установки (большой диск)
REPO    = ROOT / "pydbsr_src"                         # распакованный архив pydbsr (или клон репозитория)
CRM     = REPO / "examples" / "crm"                   # nbtools.py, run_model.py, models.py
NIST    = REPO / "docs" / "crm"                       # выгрузки NIST ASD (nist_*.csv)
DATA    = ROOT                                        # ваши файлы
XLSX    = DATA / "CrossSectionsIon_3.xlsx"            # Wang et al (2019), сечения e + Xe+
RUNS    = ROOT / "runs" / "crm"                       # результаты расчётов (большой диск)
XE2_RUN = ROOT / "runs" / "xe_plus_v2"                # готовый расчёт e + Xe+ (модель Ванга, J <= 25)
SCRATCH, SCRATCH_GB = Path.home() / "pydbsr_tmp", 5   # временные файлы dbsr_mat (быстрый диск)
CORES, MEM, HD_THREADS = 32, 100, 8                   # ядра, ГБ памяти, потоки диагонализации
TE = [0.3, 0.5, 0.7, 1.0, 1.5, 2.0, 3.0, 5.0, 7.0, 10.0, 15.0, 20.0]   # сетка Te (эВ) для констант скоростей
sys.path.insert(0, str(CRM))
import numpy as np, matplotlib.pyplot as plt
import pydbsr as db
from pydbsr import rates, reference
import nbtools as nb
%matplotlib inline

In [ ]:
import shutil
JMAX, EMAX = 25, 60.0        # J: проверьте сходимость; EMAX (эВ): σ выше — хвост 1/E (как в 02)
ref = reference.read_xlsx(XLSX)
WANG_INITIAL = sorted({i for i, j in ref.sigma})         # 1, 2, 4, 5, ... 34
W = RUNS / "xe2_wang_full"
if list((XE2_RUN / "scat").glob("h.[0-9][0-9][0-9]")):
    # готовый расчёт xe_plus_v2 (examples/xe_plus_wang2019.ipynb) — та же модель, сетка энергий
    # 0.0136 эВ до 60 эВ: рассеяние не пересчитывается, считаются только A, σ и k(Te)
    W = XE2_RUN
    om = sorted((W / "compare").glob(f"omega_J{JMAX:g}_E*.npz"))
    if om and not (W / f"omega_J{JMAX:g}.npz").exists():
        (W / f"omega_J{JMAX:g}.npz").symlink_to(om[0])
    print("готовый расчёт:", W, "| парциальных волн:", len(list((W / "scat").glob("h.[0-9][0-9][0-9]"))),
          "| Ω:", (W / f"omega_J{JMAX:g}.npz").resolve())
elif (RUNS / "xe2_wang" / "target").exists() and not (W / "target").exists():
    shutil.copytree(RUNS / "xe2_wang" / "target", W / "target")      # мишень из ноутбука 01
job = nb.Job(W, "xe2_wang", levels=XLSX, stage="all", jmax=JMAX, cores=CORES, mem=MEM, hd_threads=HD_THREADS,
             scratch=SCRATCH, scratch_gb=SCRATCH_GB, emax=EMAX, de=0.0136, multipoles="E1",
             sigma_initial=",".join(map(str, WANG_INITIAL)), te=",".join(map(str, TE)))
job.start()

In [ ]:
job.status()          # ход расчёта (запускайте когда хотите)

In [ ]:
cs = db.CollisionStrengths.load(W / f"omega_J{JMAX:g}.npz")
tg = db.Target.load(W / "target")
name = {s.nist_no: s.name for s in tg.states if s.nist_no and s.name in cs.names}   # номер Ванга -> состояние
Ecm = {lv.no: lv.energy_cm for lv in ref.levels}
def dbsr_sigma(i, j):
    E, s = cs.incident_energy(name[i]), cs.sigma(name[i], name[j])
    m = np.isfinite(s) & (E > 0)
    return E[m], s[m]
def thr(i, j):
    return (Ecm[j] - Ecm[i]) / nb.CM_PER_EV

TEa = np.array(TE)
def k_of(E, s, thr_ev):
    """Maxwellian <sigma v> (cm3/s) on the TE grid; tail above the last energy: 1/E."""
    return rates.rates_on_grid(E, s, TEa, thr_ev)
def show_k(title, k):
    print(f"{title:28s} " + "  ".join(f"{v:9.2e}" for v in k))
print(" " * 28 + "  ".join(f"Te={t:<6g}" for t in TEa))

## 1. σ Ванга против своего расчёта

Для каждой пары: k_DBSR/k_Ванг при каждом Te (обе константы — из σ одной и той же функцией
`rates.rates_on_grid`, хвост 1/E). Сводка — медиана и доля пар в пределах ×1.5 и ×2.

In [ ]:
import csv
check = []
for (i, j), (Ew, sw) in sorted(ref.sigma.items()):
    if i in name and j in name:
        kw = k_of(Ew, sw, thr(i, j))
        kd = k_of(*dbsr_sigma(i, j), thr(i, j))
        check.append(dict(i=i, j=j, label=f"{ref.label(i)} -> {ref.label(j)}",
                          **{f"ratio_Te{t:g}": d / w for t, d, w in zip(TEa, kd, kw)}))
for t in TEa:
    q = np.array([c[f"ratio_Te{t:g}"] for c in check])
    q = q[np.isfinite(q) & (q > 0)]
    print(f"Te = {t:5g} эВ: {len(q)} пар, медиана DBSR/Ванг = {np.median(q):.2f}, "
          f"в пределах ×1.5: {np.mean((q > 1 / 1.5) & (q < 1.5)):.0%}, ×2: {np.mean((q > 0.5) & (q < 2)):.0%}")
with open(W / "check_vs_wang.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(check[0])); w.writeheader(); w.writerows(check)
print("->", W / "check_vs_wang.csv")
print("\nверхние уровни выбранных линий:")
for c in check:
    if c["j"] in nb.XE2_UPPERS:
        print(f"{c['i']:3d} -> {c['j']:3d}  " + "  ".join(f"{c[f'ratio_Te{t:g}']:5.2f}" for t in TEa))

In [ ]:
fig, axs = plt.subplots(2, len(nb.XE2_LINES), figsize=(4 * len(nb.XE2_LINES), 6.5), squeeze=False)
for col, x in enumerate(nb.XE2_LINES):
    u = x["upper"]
    for row, i in enumerate((1, 2)):
        ax = axs[row][col]
        if (i, u) in ref.sigma:
            E, s = ref.sigma[(i, u)]; ax.plot(E, s / 1e-16, "k", lw=0.8, label="Ванг 2019")
        E, s = dbsr_sigma(i, u); ax.plot(E, s / 1e-16, "r", lw=0.6, label=f"DBSR, J ≤ {JMAX}")
        ax.set_xscale("log"); ax.set_title(f"{i} → {u} ({x['wl']} нм)", fontsize=9)
axs[0][0].legend(fontsize=7); fig.supxlabel("E, эВ"); fig.supylabel("σ, 10⁻¹⁶ см²"); fig.tight_layout(); plt.show()

## 2. Времена жизни (E1) и метастабили

`TAU_META` — порог: уровни ниже 6p с τ(E1) длиннее него (или без E1-распада) считаются
метастабилями. Уровень 2 (²P₁/₂) распадается только M1 — он всегда в списке.

In [ ]:
TAU_META = 1e-6            # с
br, tot, A = nb.branching_from_csv(W / "transitions_E1.csv")
first6p = min(n for n in name if ")6p" in ref.label(n))
low = [n for n in sorted(name) if Ecm[n] < Ecm[first6p]]
META = [n for n in low if n > 1 and (n not in tot or 1 / tot[n] > TAU_META)]
rows = []
for n in sorted(name):
    tau = 1 / tot[n] if n in tot else float("inf")
    rows.append(dict(no=n, label=ref.label(n), E_eV=Ecm[n] / nb.CM_PER_EV, tau_E1_s=tau, metastable=n in META))
    if n in low:
        print(f"{n:3d} {ref.label(n):34s} {Ecm[n] / nb.CM_PER_EV:7.3f} эВ   τ(E1) = {tau:9.3g} с" + ("   метастабиль" if n in META else ""))
with open(W / "xe2_levels_tau.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0])); w.writeheader(); w.writerows(rows)
print("метастабили:", META, " (начальные уровни у Ванга:", WANG_INITIAL, ")")

## 3. σ и ⟨σv⟩ между основным, ²P₁/₂ и метастабилями

Набор M — 1, 2, метастабили и начальные уровни Ванга ниже 6p (у Ванга это и 6s/5d с
τ ≈ 0.1–0.2 мкс, которые в плазме тоже заметно заселены). Только возбуждение i → j (E_i < E_j). Обратный процесс — детальное равновесие:
`rates.deexcitation(k, g_i, g_j, ΔE, Te)`, g = 2J+1 (столбец J в «NIST Level Table»).
Плюс k(i → u) из этих уровней в верхние уровни выбранных линий (у Ванга есть не все).

In [ ]:
M = sorted({1, 2} | set(META) | {i for i in WANG_INITIAL if Ecm[i] < Ecm[first6p]})   # + начальные уровни Ванга ниже 6p
sig_m, lab_m, k_m = {}, {}, {}
for a_, i in enumerate(M):
    for j in M[a_ + 1:] + [u for u in nb.XE2_UPPERS]:
        if Ecm[j] <= Ecm[i] or (i, j) in sig_m:
            continue
        E, s = dbsr_sigma(i, j)
        sig_m[(i, j)] = (E, s)
        lab_m[(i, j)] = f"{i}->{j}" + ("" if (i, j) in ref.sigma or j not in nb.XE2_UPPERS else " (нет у Ванга)")
        k_m[(i, j)] = k_of(E, s, thr(i, j))
reference.write_xlsx(W / "xe2_sigma_metastable.xlsx", ref.levels, sig_m, sheet_of=lambda k: f"from {k[0]}", labels=lab_m)
reference.write_rates_xlsx(W / "xe2_k_metastable.xlsx", TEa, k_m, lab_m,
                           note=f"<sigma v> (cm3/s), excitation i->j, levels: Wang 2019 numbers; sigma: DBSR, model xe2_wang, J<={JMAX}")
for key in sorted(k_m):
    show_k(lab_m[key], k_m[key])

## 4. Каскады в верхние уровни выбранных линий

P(h→u) — доля распадов h, проходящих через u (ветвления DBSR E1, все цепочки). Печатаются
уровни с P > `P_MIN`. k_casc(i→u) = Σ_h k(i→h)·P(h→u); отношение к прямому k(i→u).

In [ ]:
P_MIN = 1e-3
P = nb.cascade_prob(br)
prow, k_c, lab_c = [], {}, {}
kcache = {}
def k_pair(i, j):
    if (i, j) not in kcache:
        kcache[(i, j)] = k_of(*dbsr_sigma(i, j), thr(i, j))
    return kcache[(i, j)]
for x in nb.XE2_LINES:
    u = x["upper"]
    feed = sorted(((h, P[h][u]) for h in P if P[h].get(u, 0) > P_MIN and h in name), key=lambda t: -t[1])
    print(f"\n{x['wl']} нм, верхний уровень {u} {ref.label(u)}: питающих уровней {len(feed)}")
    for h, p in feed:
        prow.append(dict(upper=u, line_nm=x["wl"], h=h, h_label=ref.label(h), P=p))
        print(f"   {h:3d} {ref.label(h):34s} P = {p:.3f}")
    for i in M:
        if Ecm[i] >= Ecm[u]:
            continue
        kc = sum(k_pair(i, h) * p for h, p in feed if Ecm[h] > Ecm[i])
        kd = k_pair(i, u)
        k_c[(i, u, "casc")] = kc; lab_c[(i, u, "casc")] = f"{i}->{u} cascade"
        k_c[(i, u, "ratio")] = kc / kd; lab_c[(i, u, "ratio")] = f"{i}->{u} cascade/direct"
        if i in (1, 2):
            show_k(f"  из {i}: k_casc/k_прям", kc / kd)
with open(W / "xe2_cascade_P.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(prow[0])); w.writeheader(); w.writerows(prow)
reference.write_rates_xlsx(W / "xe2_k_cascade.xlsx", TEa, k_c, lab_c,
                           note="cascade: sum_h <sigma v>(i->h) P(h->u), cm3/s; ratio = cascade/direct; DBSR, model xe2_wang")
print("->", W / "xe2_cascade_P.csv", W / "xe2_k_cascade.xlsx")